# 11 — Incident Response & Digital Forensics

**Author:** Bency (Benjamin Adjei)  
**Course:** M.S. Cybersecurity — Incident Response & Digital Forensics  
**Tools:** Python 3, os, hashlib, datetime, json, re

---

## Objectives
- Understand the Incident Response (IR) lifecycle
- Triage and classify security incidents
- Collect and preserve digital evidence (chain of custody)
- Perform timeline analysis from system artifacts
- Analyse memory and disk artifacts
- Write a professional IR report

> 🔒 **Forensic Principle:** Never work on original evidence. Always create a verified hash of the original and work from a copy.

## 1. Setup & Imports

In [ ]:
import os
import re
import json
import hashlib
import platform
from datetime import datetime, timezone
from pathlib import Path
from collections import defaultdict, Counter

## 2. Incident Response Lifecycle

```
  1. Preparation  →  2. Detection & Analysis  →  3. Containment
                                                        ↓
  6. Lessons Learned  ←  5. Post-Incident  ←  4. Eradication & Recovery
```

### NIST SP 800-61 IR Phases

| Phase | Key Activities |
|-------|---------------|
| **Preparation** | IR plan, playbooks, tools, training |
| **Detection & Analysis** | Alerts, log review, triage, scope assessment |
| **Containment** | Isolate affected systems, block IOCs, preserve evidence |
| **Eradication** | Remove malware, patch vulnerability, reset credentials |
| **Recovery** | Restore from clean backup, monitor for recurrence |
| **Post-Incident** | Root cause analysis, lessons learned, report |

### Incident Severity Levels

| Severity | Description | Response Time |
|----------|-------------|---------------|
| **P1 – Critical** | Active breach, data exfiltration, ransomware | Immediate (< 1 hr) |
| **P2 – High** | Confirmed compromise, lateral movement detected | < 4 hours |
| **P3 – Medium** | Suspicious activity, possible intrusion attempt | < 24 hours |
| **P4 – Low** | Policy violation, isolated anomaly | < 72 hours |

## 3. Incident Triage & Classification

In [ ]:
# Simulated security alert from SIEM
INCOMING_ALERT = {
    'alert_id'   : 'INC-2025-0042',
    'timestamp'  : '2025-01-16T02:34:17Z',
    'source'     : 'Microsoft Sentinel',
    'rule'       : 'Possible Lateral Movement via PsExec',
    'host'       : 'WORKSTATION-07',
    'user'       : 'jsmith',
    'src_ip'     : '192.168.1.45',
    'dst_ip'     : '192.168.1.100',
    'process'    : 'psexec.exe',
    'description': 'PsExec execution detected from non-admin workstation to domain controller',
    'raw_events' : 3
}


def triage_incident(alert: dict) -> dict:
    """Triage an incoming alert and assign severity + initial actions."""
    HIGH_RISK_PROCESSES = {'psexec.exe', 'mimikatz.exe', 'procdump.exe',
                           'mshta.exe', 'wscript.exe', 'certutil.exe', 'bitsadmin.exe'}
    LATERAL_MOVEMENT_RULES = {'psexec', 'lateral movement', 'pass-the-hash', 'wmi exec'}

    severity = 'P4'
    findings = []

    process = alert.get('process', '').lower()
    rule    = alert.get('rule', '').lower()
    desc    = alert.get('description', '').lower()

    if any(r in rule for r in LATERAL_MOVEMENT_RULES):
        severity = 'P2'
        findings.append('Lateral movement indicator detected')

    if process in HIGH_RISK_PROCESSES:
        severity = 'P1' if severity == 'P2' else 'P2'
        findings.append(f'High-risk process executed: {process}')

    if 'domain controller' in desc:
        severity = 'P1'
        findings.append('Domain Controller targeted — critical asset at risk')

    actions = {
        'P1': ['Isolate affected hosts NOW', 'Notify CISO and IR lead',
               'Preserve memory dump', 'Block src_ip at firewall'],
        'P2': ['Isolate workstation', 'Review authentication logs',
               'Check for persistence mechanisms', 'Notify security team'],
        'P3': ['Monitor host', 'Collect logs', 'Investigate user activity'],
        'P4': ['Log and monitor', 'Review during next business day']
    }

    return {
        'alert_id'      : alert['alert_id'],
        'severity'      : severity,
        'triage_time'   : datetime.now(timezone.utc).isoformat(),
        'findings'      : findings,
        'immediate_actions': actions.get(severity, [])
    }


triage = triage_incident(INCOMING_ALERT)
print(f"Incident: {triage['alert_id']}")
print(f"Severity: {triage['severity']}")
print(f"\nFindings:")
for f in triage['findings']:
    print(f'  • {f}')
print(f"\nImmediate Actions:")
for a in triage['immediate_actions']:
    print(f'  ➤ {a}')

## 4. Evidence Collection & Chain of Custody

In [ ]:
def hash_evidence(filepath: str) -> dict:
    """Hash a file for forensic evidence integrity verification."""
    results = {'file': filepath, 'exists': False}
    if not os.path.exists(filepath):
        return results
    results['exists'] = True
    results['size_bytes'] = os.path.getsize(filepath)
    for algo in ['md5', 'sha1', 'sha256']:
        h = hashlib.new(algo)
        with open(filepath, 'rb') as f:
            for chunk in iter(lambda: f.read(8192), b''):
                h.update(chunk)
        results[algo] = h.hexdigest()
    return results


def create_chain_of_custody(evidence_items: list, investigator: str, case_id: str) -> dict:
    """Create a formal chain of custody record."""
    return {
        'case_id'       : case_id,
        'investigator'  : investigator,
        'collection_time': datetime.now(timezone.utc).isoformat(),
        'system_info'   : {
            'hostname': platform.node(),
            'os'      : platform.system(),
            'version' : platform.version()[:60]
        },
        'evidence'      : evidence_items,
        'integrity_note': 'All hashes computed at time of collection. Do not modify originals.'
    }


# Demo — hash simulated evidence files (using this notebook itself as demo)
demo_files = [
    '08-notebooks/06_packet_analysis.ipynb',
    '08-notebooks/07_log_analysis_siem.ipynb'
]

evidence_items = []
for f in demo_files:
    # Simulate — create hash of the filename string as stand-in
    item = {
        'artifact'   : f,
        'type'       : 'Jupyter Notebook',
        'sha256'     : hashlib.sha256(f.encode()).hexdigest(),
        'note'       : 'Simulated hash for demonstration'
    }
    evidence_items.append(item)
    print(f"  Collected: {f}")
    print(f"  SHA-256:   {item['sha256']}\n")

coc = create_chain_of_custody(evidence_items, 'Bency', 'INC-2025-0042')
print('Chain of Custody created:')
print(json.dumps(coc, indent=2))

## 5. Timeline Reconstruction

In [ ]:
# Simulated timeline events from multiple log sources
RAW_EVENTS = [
    {'time': '2025-01-16T01:15:00Z', 'source': 'VPN',       'event': 'Successful VPN login from unusual location (Eastern Europe) for user jsmith'},
    {'time': '2025-01-16T01:22:00Z', 'source': 'AD',        'event': 'jsmith authenticated to WORKSTATION-07'},
    {'time': '2025-01-16T01:45:00Z', 'source': 'EDR',       'event': 'mimikatz.exe executed on WORKSTATION-07 (process hollowing detected)'},
    {'time': '2025-01-16T01:47:00Z', 'source': 'AD',        'event': 'Failed Kerberos TGT request for 15 service accounts (possible AS-REP roasting)'},
    {'time': '2025-01-16T01:52:00Z', 'source': 'Firewall',  'event': 'Outbound connection from WORKSTATION-07 to 45.33.32.156:443 (known C2 IP)'},
    {'time': '2025-01-16T02:10:00Z', 'source': 'AD',        'event': 'New local admin account "svc_backup2" created on DC-01'},
    {'time': '2025-01-16T02:34:00Z', 'source': 'SIEM',      'event': 'PsExec execution detected from WORKSTATION-07 to DC-01'},
    {'time': '2025-01-16T02:40:00Z', 'source': 'EDR',       'event': 'Large file archive (backup.zip, 4.2GB) created in C:\\Temp on DC-01'},
    {'time': '2025-01-16T02:55:00Z', 'source': 'Firewall',  'event': 'DNS tunnelling pattern detected — 847 TXT record queries in 60s from DC-01'},
    {'time': '2025-01-16T03:01:00Z', 'source': 'DLP',       'event': 'Large data transfer (4.1GB) to external IP 45.33.32.156 blocked'},
]

ATTACK_PHASES = {
    'VPN'     : 'Initial Access',
    'AD'      : 'Credential Access / Lateral Movement',
    'EDR'     : 'Execution / Defense Evasion',
    'Firewall': 'Command & Control',
    'SIEM'    : 'Lateral Movement',
    'DLP'     : 'Exfiltration'
}

print('=== INCIDENT TIMELINE (MITRE ATT&CK Mapped) ===\n')
for event in sorted(RAW_EVENTS, key=lambda x: x['time']):
    phase = ATTACK_PHASES.get(event['source'], 'Unknown')
    print(f"  {event['time']}  [{event['source']:<9}]  [{phase:<30}]")
    print(f"    {event['event']}\n")

## 6. Indicators of Compromise (IOC) Extraction

In [ ]:
def extract_iocs(events: list) -> dict:
    """Extract IOCs (IPs, filenames, hashes, usernames) from event descriptions."""
    iocs = defaultdict(set)

    IP_RE   = re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b')
    FILE_RE = re.compile(r'\b[\w\-\.]+\.(?:exe|zip|bat|ps1|dll|vbs|js|py)\b', re.I)
    USER_RE = re.compile(r'\bfor user (\w+)|\b(jsmith|svc_\w+)\b', re.I)
    HOST_RE = re.compile(r'\b(WORKSTATION-\d+|DC-\d+)\b')

    for event in events:
        text = event['event'] + ' ' + event['source']
        iocs['ip_addresses'].update(IP_RE.findall(text))
        iocs['filenames'].update(FILE_RE.findall(text))
        for m in USER_RE.finditer(text):
            iocs['usernames'].update(filter(None, m.groups()))
        iocs['hostnames'].update(HOST_RE.findall(text))

    return {k: sorted(v) for k, v in iocs.items()}


iocs = extract_iocs(RAW_EVENTS)
print('=== EXTRACTED IOCs ===\n')
for category, values in iocs.items():
    print(f'  {category.upper()}:')
    for v in values:
        print(f'    • {v}')
    print()

## 7. MITRE ATT&CK Mapping

In [ ]:
MITRE_MAPPING = [
    {'tactic': 'Initial Access',        'technique': 'T1133', 'name': 'External Remote Services',         'evidence': 'VPN login from Eastern Europe'},
    {'tactic': 'Execution',             'technique': 'T1059', 'name': 'Command and Scripting Interpreter', 'evidence': 'PsExec remote execution'},
    {'tactic': 'Credential Access',     'technique': 'T1003', 'name': 'OS Credential Dumping',            'evidence': 'mimikatz.exe execution'},
    {'tactic': 'Credential Access',     'technique': 'T1558', 'name': 'Steal or Forge Kerberos Tickets',  'evidence': 'AS-REP roasting attempts'},
    {'tactic': 'Lateral Movement',      'technique': 'T1570', 'name': 'Lateral Tool Transfer via PsExec', 'evidence': 'PsExec to DC-01'},
    {'tactic': 'Persistence',           'technique': 'T1136', 'name': 'Create Account',                   'evidence': 'New admin account svc_backup2 on DC-01'},
    {'tactic': 'Command & Control',     'technique': 'T1071', 'name': 'Application Layer Protocol (DNS)', 'evidence': 'DNS tunnelling from DC-01'},
    {'tactic': 'Exfiltration',          'technique': 'T1048', 'name': 'Exfiltration Over Alt Protocol',   'evidence': '4.1GB transfer to external IP (blocked)'},
]

print('=== MITRE ATT&CK MAPPING ===\n')
print(f'{"Tactic":<25} {"Technique":<10} {"Name":<40} Evidence')
print('-' * 120)
for t in MITRE_MAPPING:
    print(f"{t['tactic']:<25} {t['technique']:<10} {t['name']:<40} {t['evidence']}")

## 8. Incident Response Report

In [ ]:
ir_report = {
    'case_id'           : 'INC-2025-0042',
    'report_generated'  : datetime.now(timezone.utc).isoformat(),
    'investigator'      : 'Bency (Benjamin Adjei)',
    'incident_summary'  : {
        'title'         : 'Targeted Intrusion — Credential Theft & Attempted Data Exfiltration',
        'severity'      : triage['severity'],
        'status'        : 'Contained',
        'affected_hosts': ['WORKSTATION-07', 'DC-01'],
        'affected_users': ['jsmith'],
        'attack_duration': '1h 46m (01:15Z – 03:01Z)',
        'data_exfiltrated': 'Blocked — 4.1GB transfer prevented by DLP'
    },
    'timeline_events'   : len(RAW_EVENTS),
    'iocs'              : iocs,
    'mitre_techniques'  : [f"{t['technique']} — {t['name']}" for t in MITRE_MAPPING],
    'containment_actions': [
        'WORKSTATION-07 isolated from network at 02:35Z',
        'jsmith account disabled and password reset',
        'svc_backup2 account deleted from DC-01',
        'C2 IP 45.33.32.156 blocked at perimeter firewall',
        'DNS queries to suspicious domains blocked'
    ],
    'root_cause'        : 'Phishing email delivered credential harvester; jsmith credentials compromised',
    'recommendations'   : [
        'Enable MFA on VPN and all privileged accounts',
        'Deploy Privileged Access Workstations (PAWs) for admin tasks',
        'Enable Credential Guard on Windows systems to block mimikatz',
        'Restrict PsExec via AppLocker/WDAC policy',
        'Implement DNS filtering to detect/block tunnelling',
        'Conduct phishing awareness training for all staff'
    ],
    'chain_of_custody'  : coc
}

print(json.dumps(ir_report, indent=2))

## 9. Key Takeaways

| Concept | Takeaway |
|---------|----------|
| **Chain of custody** | Hash every artifact at collection — integrity is non-negotiable in court |
| **Timeline analysis** | Chronological ordering reveals attack progression and dwell time |
| **MITRE ATT&CK** | Maps adversary behaviour to a universal language for sharing TTPs |
| **IOC extraction** | Automate extraction — manual review misses patterns at scale |
| **Triage speed** | First 30 minutes determine containment success — automate triage |
| **Lessons learned** | Every incident should generate at least one control improvement |

### Key Forensic Tools (Reference)
| Tool | Use Case |
|------|----------|
| **Autopsy / FTK** | Disk image analysis |
| **Volatility** | Memory forensics |
| **Wireshark** | Network packet capture analysis |
| **Splunk / Sentinel** | Log correlation and timeline building |
| **MITRE ATT&CK Navigator** | Visualise technique coverage |
| **TheHive + Cortex** | Incident case management + automated analysis |

---
*Next: 12 — Cloud Security & AWS Fundamentals*